Fabric Notebook: Generate Customer Burst
-----------------------------------------
In-browser version of `generate_demo_data.py --mode burst`.
Run the final cell during the live demo to simulate ~25 new customers
arriving on the site. Writes events_ingest, session_signals,
next_best_actions, and session_state docs directly into Atlas.

Sync note: generation logic is mirrored from generate_demo_data.py.
If that file changes, update this notebook to match.

Copy each cell block into a separate Fabric notebook cell.

In [ ]:
# ===== Cell 1: Install pymongo (first run only) =====

%pip install pymongo


In [ ]:
# ===== Cell 2: Parameters (mark as "Toggle parameter cell" in Fabric) =====

MONGODB_URI = "<YOUR_MONGODB_URI>"  # Replace with your Atlas connection string
DB_NAME = "leafy_popup_store"
BURST_USER_COUNT = 25


In [ ]:
# ===== Cell 3: Imports and constants =====

import random
import uuid
from datetime import datetime, timedelta

from bson import ObjectId
from pymongo import MongoClient

ARCHETYPES = {
    "loyal_shopper":    {"weight": 0.25, "signals": {"high-intent": 0.60, "search-friction": 0.10, "exit-risk": 0.30}, "signal_count": (8, 20), "redeem_rate": (0.60, 0.80), "sessions": (2, 5)},
    "window_shopper":   {"weight": 0.20, "signals": {"high-intent": 0.20, "search-friction": 0.50, "exit-risk": 0.30}, "signal_count": (5, 15), "redeem_rate": (0.10, 0.20), "sessions": (1, 3)},
    "bouncing_visitor": {"weight": 0.20, "signals": {"high-intent": 0.10, "search-friction": 0.10, "exit-risk": 0.80}, "signal_count": (3, 8),  "redeem_rate": (0.00, 0.10), "sessions": (1, 2)},
    "discount_hunter":  {"weight": 0.15, "signals": {"high-intent": 0.15, "search-friction": 0.35, "exit-risk": 0.50}, "signal_count": (6, 16), "redeem_rate": (0.40, 0.60), "sessions": (2, 4)},
    "engaged_browser":  {"weight": 0.20, "signals": {"high-intent": 0.40, "search-friction": 0.35, "exit-risk": 0.25}, "signal_count": (6, 18), "redeem_rate": (0.30, 0.50), "sessions": (2, 5)},
}

HIGH_INTENT_EVIDENCE = [
    "Customer viewed {product} {n} times in the last {m} minutes and added it to their wishlist",
    "Repeated views on {product} with prolonged engagement, suggesting strong purchase intent",
    "User showed focused attention on {product}, spending over {m} minutes on the product page",
    "Customer returned to {product} across {n} page visits, indicating high purchase likelihood",
    "Strong engagement signals: {n} views of {product} with image zoom and size selection",
]

SEARCH_FRICTION_EVIDENCE = [
    "User searched for '{query}' with no clear results, browsing {n} products without convergence",
    "Multiple searches for '{query}' variants with scattered attention across categories",
    "Customer explored {n} products after searching '{query}' but showed no add-to-cart intent",
    "Repeated search refinements around '{query}' suggest difficulty finding the right product",
]

EXIT_RISK_EVIDENCE = [
    "Customer idle for {n} seconds after viewing {m} products, cursor moved toward browser close",
    "Tab switch detected after {m} minutes of browsing, engagement dropping rapidly",
    "User scrolled to page bottom and stopped interacting for {n} seconds",
    "Cart abandoned after {n} seconds of inactivity, no further product views",
]

SEARCH_QUERIES = [
    "running shoes", "summer dress", "leather jacket", "cotton t-shirt",
    "winter coat", "hiking boots", "yoga pants", "denim jeans",
    "silk blouse", "casual sneakers", "formal shirt", "rain jacket",
    "bath towels", "shower curtain", "bed sheets", "kitchen organizer",
]

TOPIC_DIMENSIONS = [
    {"dimension": "articleType", "values": ["Shoes", "Dress", "Shirt", "Jacket", "Pants"]},
    {"dimension": "color",       "values": ["Black", "Blue", "Red", "White", "Green", "Beige"]},
    {"dimension": "subCategory", "values": ["Bath", "Hardware", "Nursery", "Home Furnishing", "Accessories"]},
]

EXIT_METHODS = ["logout-hover", "tab-switch", "idle-timeout", "back-button", "close-button"]

SOCIAL_PROOF_TITLES = ["Popular Right Now", "Trending Pick", "Good Choice", "Hot Item", "Customers Love This"]
SOCIAL_PROOF_MESSAGES = [
    "{n} customers purchased {category} recently.",
    "This {category} item is trending with {n} purchases this week.",
    "{n} shoppers added {category} items to their cart today.",
]
SOCIAL_PROOF_EMBED_MESSAGES = ["This item is trending", "High demand item", "Popular with shoppers", "Selling fast"]

DISCOUNT_TITLES = ["Special Offer", "Just For You", "Personalized Deal", "Smart Match"]
DISCOUNT_MESSAGES = [
    "Based on your interests, get 10% off {product} today!",
    "We think you'll love {product}. Enjoy 15% off!",
    "Personalized pick: {product} at 20% off for a limited time.",
]

SHIPPING_DISCOUNT_CONFIG = {
    "low":    {"titles": ["Before you go", "Wait a moment"],           "messages": ["Get 5% off your shipping if you complete your purchase today."]},
    "medium": {"titles": ["Don't miss out", "Limited time offer"],      "messages": ["Complete your order now and get 15% off shipping."]},
    "high":   {"titles": ["Wait! Special offer", "Exclusive deal"],     "messages": ["Complete your order now and get 50% off shipping!"]},
    "urgent": {"titles": ["Last chance", "Final offer"],                "messages": ["Complete your order now and get FREE shipping!"]},
}

EVENT_WEIGHTS = {"search": 1, "view-product": 3, "add-to-cart": 7}
COUNTED_EVENTS = ("heartbeat", "search", "view-product", "add-to-cart", "exit-risk")

MODE = "burst"


In [ ]:
# ===== Cell 4: Helper functions (event + NBA generation) =====

def pick_archetype():
    names = list(ARCHETYPES.keys())
    weights = [ARCHETYPES[n]["weight"] for n in names]
    return random.choices(names, weights=weights)[0]

def pick_severity(signal_type):
    if signal_type == "exit-risk":
        return random.choices(["low", "medium", "high", "urgent"], weights=[0.20, 0.30, 0.30, 0.20])[0]
    if signal_type == "search-friction":
        return random.choices(["low", "medium", "high", "urgent"], weights=[0.25, 0.35, 0.30, 0.10])[0]
    return random.choices(["low", "medium", "high", "urgent"], weights=[0.15, 0.30, 0.40, 0.15])[0]

def burst_timestamp(session_offset_minutes=0):
    # Last 1-5 minutes, so new data appears fresh in the demo.
    offset = random.uniform(60, 300) - session_offset_minutes * 60
    return datetime.utcnow() - timedelta(seconds=max(offset, 10))

def generate_evidence(signal_type, product_name=None, query=None):
    n, m = random.randint(2, 12), random.randint(2, 15)
    if signal_type == "high-intent":
        return random.choice(HIGH_INTENT_EVIDENCE).format(product=product_name or "a product", n=n, m=m)
    if signal_type == "search-friction":
        return random.choice(SEARCH_FRICTION_EVIDENCE).format(query=query or "items", n=n)
    return random.choice(EXIT_RISK_EVIDENCE).format(n=n * 5, m=m)

def make_event(uid, sid, event_name, metadata, event_ts):
    return {
        "tags": {"userId": uid, "sessionId": sid, "event": event_name},
        "timestamp": event_ts,
        "metadata": metadata,
        "_demo_source": MODE,
    }

def generate_raw_events(signal_type, uid, sid, product, query, ts, products):
    events = []
    num_heartbeats = random.randint(2, 6)
    for i in range(num_heartbeats):
        events.append(make_event(uid, sid, "heartbeat", {}, ts - timedelta(seconds=(num_heartbeats - i) * 10)))

    if signal_type == "high-intent":
        for i in range(random.randint(3, 8)):
            vp = product if product else (random.choice(products) if products else None)
            if vp:
                events.append(make_event(uid, sid, "view-product", {
                    "productId": str(vp["_id"]), "subCategory": vp.get("subCategory", ""),
                    "articleType": vp.get("articleType", ""), "brand": vp.get("brand", ""),
                }, ts - timedelta(seconds=(8 - i) * random.randint(8, 30))))
        if product and random.random() < 0.4:
            events.append(make_event(uid, sid, "add-to-cart", {
                "productId": str(product["_id"]), "subCategory": product.get("subCategory", ""),
                "articleType": product.get("articleType", ""), "brand": product.get("brand", ""),
            }, ts - timedelta(seconds=random.randint(2, 10))))

    elif signal_type == "search-friction":
        queries = random.sample(SEARCH_QUERIES, min(random.randint(2, 5), len(SEARCH_QUERIES)))
        for i, sq in enumerate(queries):
            sp = random.choice(products) if products else None
            search_ts = ts - timedelta(seconds=(len(queries) - i) * random.randint(15, 45))
            events.append(make_event(uid, sid, "search", {
                "query": sq,
                "productId": str(sp["_id"]) if sp else None,
                "subCategory": sp.get("subCategory", "") if sp else None,
                "articleType": sp.get("articleType", "") if sp else None,
                "brand": sp.get("brand", "") if sp else None,
            }, search_ts))
            for j in range(random.randint(1, 3)):
                vp = random.choice(products) if products else None
                if vp:
                    events.append(make_event(uid, sid, "view-product", {
                        "productId": str(vp["_id"]), "subCategory": vp.get("subCategory", ""),
                        "articleType": vp.get("articleType", ""), "brand": vp.get("brand", ""),
                    }, search_ts + timedelta(seconds=random.randint(3, 12))))

    elif signal_type == "exit-risk":
        for i in range(random.randint(1, 4)):
            vp = random.choice(products) if products else None
            if vp:
                events.append(make_event(uid, sid, "view-product", {
                    "productId": str(vp["_id"]), "subCategory": vp.get("subCategory", ""),
                    "articleType": vp.get("articleType", ""), "brand": vp.get("brand", ""),
                }, ts - timedelta(seconds=(4 - i) * random.randint(10, 25))))
        events.append(make_event(uid, sid, "exit-risk", {"exitMethod": random.choice(EXIT_METHODS)},
                                 ts - timedelta(seconds=random.randint(1, 5))))

    events.sort(key=lambda e: e["timestamp"])
    return events

def generate_nba(signal_type, severity, uid, sid, product, ts, should_redeem):
    triggered_by = f"{severity}_{signal_type}"
    if signal_type == "high-intent":
        nba = {
            "uid": uid, "sid": sid, "type": "social-proof-notification",
            "actionMetadata": {
                "title": random.choice(SOCIAL_PROOF_TITLES),
                "message": random.choice(SOCIAL_PROOF_MESSAGES).format(
                    n=random.randint(3, 25),
                    category=product.get("articleType", "this category") if product else "this category"),
                "triggeredBySignal": triggered_by,
            },
            "ts": ts + timedelta(seconds=random.randint(1, 5)),
            "redeemed": should_redeem, "_demo_source": MODE,
        }
        if product and random.random() < 0.7:
            nba["embedInProduct"] = {"productId": str(product["_id"]), "message": random.choice(SOCIAL_PROOF_EMBED_MESSAGES)}
        return nba
    if signal_type == "search-friction":
        nba = {
            "uid": uid, "sid": sid, "type": "discount-product-recommendation",
            "actionMetadata": {
                "title": random.choice(DISCOUNT_TITLES),
                "message": random.choice(DISCOUNT_MESSAGES).format(product=product.get("name", "this product") if product else "this product"),
                "triggeredBySignal": triggered_by,
            },
            "ts": ts + timedelta(seconds=random.randint(2, 8)),
            "redeemed": should_redeem, "_demo_source": MODE,
        }
        if product:
            nba["actionMetadata"]["productRecommendation"] = {
                "productId": str(product["_id"]), "name": product.get("name", ""),
                "imageUrl": product.get("image", {}).get("url", ""),
            }
        return nba
    cfg = SHIPPING_DISCOUNT_CONFIG.get(severity, SHIPPING_DISCOUNT_CONFIG["low"])
    return {
        "uid": uid, "sid": sid, "type": "shipping-discount",
        "actionMetadata": {
            "title": random.choice(cfg["titles"]),
            "message": random.choice(cfg["messages"]),
            "triggeredBySignal": triggered_by,
        },
        "ts": ts + timedelta(seconds=random.randint(1, 3)),
        "redeemed": should_redeem, "_demo_source": MODE,
    }


In [ ]:
# ===== Cell 5: session_state builder (mirrors asp1_session_state_builder) =====

def _event_priority(event_name):
    return 1 if event_name == "exit-risk" else 0

def compute_intent(window_events):
    products_by_id, articles_by_type, subcats_by_name = {}, {}, {}
    p_tot = a_tot = s_tot = 0
    for e in window_events:
        w = EVENT_WEIGHTS.get(e["tags"]["event"], 0)
        if w <= 0: continue
        ev = e["tags"]["event"]
        meta = e.get("metadata") or {}
        pid, at, sc = meta.get("productId"), meta.get("articleType"), meta.get("subCategory")
        for key, buckets, totv in ((pid, products_by_id, "p"), (at, articles_by_type, "a"), (sc, subcats_by_name, "s")):
            if not key: continue
            r = buckets.setdefault(key, {"searchCount": 0, "viewCount": 0, "addToCartCount": 0, "weight": 0})
            if totv == "p": r["productId"] = key
            elif totv == "a": r["articleType"] = key
            else: r["subCategory"] = key
            if ev == "search": r["searchCount"] += 1
            elif ev == "view-product": r["viewCount"] += 1
            elif ev == "add-to-cart": r["addToCartCount"] += 1
            r["weight"] += w
            if totv == "p": p_tot += w
            elif totv == "a": a_tot += w
            else: s_tot += w

    def finalize(buckets, total):
        out = list(buckets.values())
        for r in out:
            r["focus"] = (r["weight"] / total) if total > 0 else 0
        return out

    return {
        "products": finalize(products_by_id, p_tot),
        "articleTypes": finalize(articles_by_type, a_tot),
        "subCategories": finalize(subcats_by_name, s_tot),
        "dimensionTotals": {"productWeightTotal": p_tot, "articleTypeWeightTotal": a_tot, "subCategoryWeightTotal": s_tot},
    }

def build_session_state(uid, sid, events):
    if not events: return None
    events_sorted = sorted(events, key=lambda e: e["timestamp"])
    windows = {}
    event_counts = {n: 0 for n in COUNTED_EVENTS}
    for e in events_sorted:
        name = e["tags"]["event"]
        if name in event_counts: event_counts[name] += 1
        ws = (int(e["timestamp"].timestamp()) // 10) * 10
        windows.setdefault(ws, []).append(e)
    last_ws = max(windows.keys())
    last_events = windows[last_ws]
    best = max(last_events, key=lambda e: (_event_priority(e["tags"]["event"]), e["timestamp"]))
    last_event_obj = {"event": best["tags"]["event"], "ts": best["timestamp"]}
    search_history = [e.get("metadata", {}).get("query") for e in events_sorted
                      if e["tags"]["event"] == "search" and e.get("metadata", {}).get("query")]
    return {
        "userId": uid, "sessionId": sid,
        "firstSeen": events_sorted[0]["timestamp"],
        "lastSeen": events_sorted[-1]["timestamp"],
        "lastEvent": last_event_obj,
        "last10s": {
            "intent": compute_intent(last_events),
            "lastEvent": last_event_obj,
            "window": {"start": datetime.utcfromtimestamp(last_ws), "end": datetime.utcfromtimestamp(last_ws + 10)},
        },
        "searchHistory": search_history,
        "sessionTotals": {"eventCounts": event_counts, "windowCount": len(windows)},
        "_demo_source": MODE,
    }


### Cell 6 is self-contained

Parameters (`MONGODB_URI`, `DB_NAME`, `BURST_USER_COUNT`) are defined at the top of Cell 6 itself, 
so it works even if `%pip install` in Cell 2 restarted the kernel. 
Edit the URI at the top of Cell 6 before running.


In [ ]:
# ===== Cell 6: RUN BURST (click-to-run cell for the demo) =====
print(f"Connecting to Atlas...")
client = MongoClient(MONGODB_URI, tls=True, tlsAllowInvalidCertificates=True)
db = client[DB_NAME]
db.command("ping")
print(f"Connected to {DB_NAME}")

products = list(db.products.find(
    {}, {"name": 1, "image.url": 1, "articleType": 1, "subCategory": 1, "brand": 1},
).limit(100))
print(f"Loaded {len(products)} products for reference")

all_events, all_signals, all_nbas, all_session_states = [], [], [], []

for _ in range(BURST_USER_COUNT):
    uid = str(ObjectId())
    arch = ARCHETYPES[pick_archetype()]
    num_sessions = random.randint(*arch["sessions"])
    total_signals = random.randint(*arch["signal_count"])
    redeem_rate = random.uniform(*arch["redeem_rate"])
    signals_per_session = max(1, total_signals // num_sessions)

    for sess_idx in range(num_sessions):
        sid = str(uuid.uuid4())
        session_events = []
        n_signals = (total_signals - signals_per_session * (num_sessions - 1)
                     if sess_idx == num_sessions - 1 else signals_per_session)
        n_signals = max(1, n_signals)

        for sig_idx in range(n_signals):
            signal_type = random.choices(list(arch["signals"].keys()),
                                         weights=list(arch["signals"].values()))[0]
            severity = pick_severity(signal_type)
            product = random.choice(products) if products else None
            query = random.choice(SEARCH_QUERIES)
            ts = burst_timestamp(session_offset_minutes=sig_idx * 0.5)

            signal = {
                "uid": uid, "sid": sid, "signal": signal_type, "severity": severity,
                "evidence": generate_evidence(signal_type, product.get("name") if product else None, query),
                "ts": ts, "_demo_source": MODE,
            }
            if signal_type == "high-intent" and product:
                signal["productId"] = str(product["_id"])
            elif signal_type == "search-friction":
                td = random.choice(TOPIC_DIMENSIONS)
                signal["topic"] = {"dimension": td["dimension"], "value": random.choice(td["values"])}
            all_signals.append(signal)

            raw = generate_raw_events(signal_type, uid, sid, product, query, ts, products)
            all_events.extend(raw)
            session_events.extend(raw)

            all_nbas.append(generate_nba(signal_type, severity, uid, sid, product, ts, random.random() < redeem_rate))

        ss = build_session_state(uid, sid, session_events)
        if ss: all_session_states.append(ss)

print(f"Generated: {len(all_events)} events, {len(all_signals)} signals, {len(all_nbas)} NBAs, {len(all_session_states)} session_states")
print("Inserting into Atlas...")

if all_events:          db.events_ingest.insert_many(all_events);           print(f"  events_ingest: +{len(all_events)}")
if all_signals:         db.session_signals.insert_many(all_signals);        print(f"  session_signals: +{len(all_signals)}")
if all_nbas:            db.next_best_actions.insert_many(all_nbas);         print(f"  next_best_actions: +{len(all_nbas)}")
if all_session_states:  db.session_state.insert_many(all_session_states);   print(f"  session_state: +{len(all_session_states)}")

client.close()
print("\nBurst complete! New customers are on their way into Fabric via mirroring.")
